In [ ]:

import pandas as pd
import re

# ============================================================
# 1. 输入文件
# ============================================================

gwas_file = "/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS.bed"
sequence_report_file = "/home/hnadmin/data/yangshuju/sample/sequence_report-2.tsv"

# 输出文件
output_bed = "/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_Oar_v2_NC.bed"
output_excel = "/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_Oar_v2_NC.xlsx"


# ============================================================
# 2. 读取 GWAS
# ============================================================

gwas = pd.read_excel(
    gwas_file,
    header=1
)

print("GWAS columns:")
print(gwas.columns.tolist())

print("\nGWAS chromosome:")
print(sorted(gwas["Chrome"].dropna().unique()))


# ============================================================
# 3. 读取 NCBI sequence report
# ============================================================

seq = pd.read_csv(
    sequence_report_file,
    sep="\t",
    dtype=str
)

print("\nSequence report columns:")
print(seq.columns.tolist())


# ============================================================
# 4. 查看 assembly 信息
# ============================================================

if "Assembly Accession" in seq.columns:
    print("\nAssembly accessions:")
    print(seq["Assembly Accession"].dropna().unique())


# ============================================================
# 5. 筛选染色体序列
#
# sequence report 中：
# Chromosome name -> 1, 2, 3 ... 26
# RefSeq seq accession -> NC_xxxxx.x
#
# 只保留真正的 chromosome","不使用 unlocalized/scaffold
# ============================================================

chrom_seq = seq.copy()

# 只保留有 chromosome name 和 RefSeq accession 的记录
chrom_seq = chrom_seq[
    chrom_seq["Chromosome name"].notna() &
    chrom_seq["RefSeq seq accession"].notna()
].copy()

# ============================================================
# 6. 标准化 chromosome 名称
# ============================================================

def normalize_chromosome(x):
    """
    将 chromosome name 标准化为字符串数字。
    例如：
    1 -> 1
    chr1 -> 1
    chromosome 1 -> 1
    """

    x = str(x).strip()

    # 去掉 chr
    x = re.sub(r"^chr", "", x, flags=re.IGNORECASE)

    # chromosome 14 -> 14
    x = re.sub(
        r"^chromosome[\s_]*",
        "",
        x,
        flags=re.IGNORECASE
    )

    return x


chrom_seq["chrom_norm"] = (
    chrom_seq["Chromosome name"]
    .apply(normalize_chromosome)
)


# ============================================================
# 7. 建立：
#
# GWAS chromosome
#       ↓
# RefSeq accession
#
# 例如：
# 14 -> NC_056067.1
# ============================================================

chrom_to_nc = dict(
    zip(
        chrom_seq["chrom_norm"],
        chrom_seq["RefSeq seq accession"]
    )
)

print("\nChromosome -> RefSeq accession:")
for chrom, accession in sorted(
    chrom_to_nc.items(),
    key=lambda x: (
        int(x[0]) if str(x[0]).isdigit() else 999,
        x[0]
    )
):
    print(f"{chrom}\t{accession}")


# ============================================================
# 8. 将 GWAS chromosome 转换成 RefSeq accession
# ============================================================

gwas["chrom_norm"] = (
    gwas["Chrome"]
    .apply(normalize_chromosome)
)

gwas["NC_accession"] = (
    gwas["chrom_norm"]
    .map(chrom_to_nc)
)


# ============================================================
# 9. 检查没有成功转换的 chromosome
# ============================================================

unmapped_chrom = gwas[
    gwas["NC_accession"].isna()
]["Chrome"].unique()

if len(unmapped_chrom) > 0:

    print("\nWARNING: 以下 chromosome 没有找到对应的 RefSeq accession:")

    for chrom in unmapped_chrom:
        print(chrom)

else:

    print("\n所有 GWAS chromosome 均成功转换。")


# ============================================================
# 10. 检查 Physical Location
# ============================================================

gwas["Physical Location"] = pd.to_numeric(
    gwas["Physical Location"],
    errors="coerce"
)

if gwas["Physical Location"].isna().any():

    print(
        "\nWARNING: 存在 Physical Location 无法转换成数字的 SNP:"
    )

    print(
        gwas[
            gwas["Physical Location"].isna()
        ]
    )


# ============================================================
# 11. 生成 BED
#
# BED 是 0-based, half-open
#
# SNP position = 1-based
#
# BED:
# start = position - 1
# end   = position
# ============================================================

bed = gwas[
    gwas["NC_accession"].notna() &
    gwas["Physical Location"].notna()
].copy()

bed["start"] = (
    bed["Physical Location"].astype(int) - 1
)

bed["end"] = (
    bed["Physical Location"].astype(int)
)


# ============================================================
# 12. BED 文件
#
# 第1列：NC accession
# 第2列：start
# 第3列：end
#
# 另外增加 SNP ID","方便追踪
# ============================================================

bed_output = bed[
    [
        "NC_accession",
        "start",
        "end"
    ]
].copy()

bed_output.to_csv(
    output_bed,
    sep="\t",
    header=False,
    index=False
)


# ============================================================
# 13. 保存转换后的完整 GWAS
# ============================================================

gwas.to_excel(
    output_excel,
    index=False
)


# ============================================================
# 14. 输出统计
# ============================================================

print("\n========================================")
print("转换完成")
print("========================================")

print(f"GWAS SNP 数量: {len(gwas)}")
print(f"成功转换 chromosome: {gwas['NC_accession'].notna().sum()}")
print(f"未成功转换: {gwas['NC_accession'].isna().sum()}")

print("\nBED 文件:")
print(output_bed)

print("\n转换后的 Excel:")
print(output_excel)

print("\nBED 前10行:")
print(bed_output.head(10))



In [ ]:
liftOver \
    gwas_v2.bed \
	/home/hnadmin/data/yangshuju/sample/GCF_016772045.1ToOviAri4.over.chain \
    gwas_Oar_v4.bed \
    gwas_Oar_v4.unmapped

In [ ]:
import pandas as pd


# ============================================================
# 1. 文件
# ============================================================

vcor_file = "/home/hnadmin/data/yangshuju/result/ovary/sheep_gwas_LD.vcor"

out_bed = "/home/hnadmin/data/yangshuju/result/ovary/sheep_gwas_LD_all.bed"

out_table = "/home/hnadmin/data/yangshuju/result/ovary/sheep_gwas_LD_all.tsv"


# ============================================================
# 2. 读取 vcor
# ============================================================

ld = pd.read_csv(
    vcor_file,
    sep="\t"
)

# 去掉 PLINK header 前面的 #
ld.columns = [
    str(x).lstrip("#")
    for x in ld.columns
]

print("========================================")
print("vcor columns")
print("========================================")

print(ld.columns.tolist())

print()
print("Number of LD pairs:", len(ld))


# ============================================================
# 3. 检查关键列
# ============================================================

required_columns = [
    "CHROM_A",
    "POS_A",
    "ID_A",
    "CHROM_B",
    "POS_B",
    "ID_B"
]

missing = [
    x for x in required_columns
    if x not in ld.columns
]

if len(missing) > 0:

    raise ValueError(
        "vcor 缺少以下列: "
        + ", ".join(missing)
    )


# ============================================================
# 4. 确定 R2 列
#
# PLINK 2 不同版本可能叫：
#
# UNPHASED_R2
# R2
# PHASED_R2
# ============================================================

r2_candidates = [
    "UNPHASED_R2",
    "R2",
    "PHASED_R2"
]

r2_col = None

for col in r2_candidates:

    if col in ld.columns:

        r2_col = col
        break


if r2_col is None:

    print()
    print("Warning: 没找到 R2 列")

else:

    print()
    print("R2 column:", r2_col)


# ============================================================
# 5. 构建：
#
# source SNP = ID_A
# LD SNP     = ID_B
#
# ============================================================

result = pd.DataFrame({

    "source_chr":
        ld["CHROM_A"],

    "source_pos":
        ld["POS_A"],

    "source_snp":
        ld["ID_A"],

    "ld_chr":
        ld["CHROM_B"],

    "ld_pos":
        ld["POS_B"],

    "ld_snp":
        ld["ID_B"]
})


# 添加 R2

if r2_col is not None:

    result["R2"] = ld[r2_col]


# ============================================================
# 6. 去除 self
#
# 理论上 PLINK --ld-snp-list 已经不会输出 self-comparison","
# 但这里再保险一次
# ============================================================

result = result[
    result["source_snp"]
    !=
    result["ld_snp"]
].copy()


# ============================================================
# 7. 去除重复
# ============================================================

result = result.drop_duplicates()


# ============================================================
# 8. 生成 LD SNP 的 BED
#
# BED:
#
# start = POS - 1
# end   = POS
# ============================================================

result["bed_start"] = (
    result["ld_pos"] - 1
)

result["bed_end"] = (
    result["ld_pos"]
)


# ============================================================
# 9. 输出详细表
# ============================================================

result.to_csv(
    out_table,
    sep="\t",
    index=False
)


# ============================================================
# 10. 输出 BED
#
# BED 第4列 = 源头 GWAS SNP
# 第5列 = R2
# ============================================================

bed_columns = [
    "ld_chr",
    "bed_start",
    "bed_end",
    "source_snp"
]

if r2_col is not None:

    bed_columns.append("R2")


result[
    bed_columns
].to_csv(
    out_bed,
    sep="\t",
    header=False,
    index=False
)


# ============================================================
# 11. 统计
# ============================================================

print()
print("========================================")
print("LD result")
print("========================================")

print(
    "LD pairs:",
    len(result)
)

print(
    "Unique source GWAS SNP:",
    result["source_snp"].nunique()
)

print(
    "Unique LD SNP:",
    result["ld_snp"].nunique()
)

print()
print("Output BED:")
print(out_bed)

print()
print("Output table:")
print(out_table)


# ============================================================
# 12. 查看前几行
# ============================================================

print()
print("First 20 records:")

print(
    result[
        [
            "source_snp",
            "ld_snp"
        ]
        + (
            ["R2"]
            if r2_col is not None
            else []
        )
    ].head(20).to_string(index=False)
)

GCF_016772045.1 → GCF_000298735.2

In [4]:
import pandas as pd
import re
from collections import defaultdict

# ============================================================
# 1. 文件
# ============================================================

bed_file = "/home/hnadmin/data/yangshuju/result/ovary/only_snp.bed"

gff_file = "/home/hnadmin/data/yangshuju/sample/GCF_000298735.2-GCF_016772045.1.gff"

out_bed = "/home/hnadmin/data/yangshuju/result/ovary/only_snp_v4.bed"

out_detail = "/home/hnadmin/data/yangshuju/result/ovary/only_snp_v4_mapping_detail.tsv"

out_unmapped = "/home/hnadmin/data/yangshuju/result/ovary/only_snp_v4.unmapped.tsv"


# ============================================================
# 2. 读取 BED
#
# BED:
# chromosome    0-based start    0-based end
# ============================================================

bed = pd.read_csv(
    bed_file,
    sep="\t",
    header=None,
    names=["source_chr", "bed_start", "bed_end","GWAS_snp","R2"],
    dtype={
        "source_chr": str,
        "bed_start": int,
        "bed_end": int,
		"GWAS_snp": str,
		"R2":float
    }
)

# SNP 的 1-based genomic position
bed["source_pos"] = bed["bed_start"] + 1

print("========================================")
print("Input BED")
print("========================================")
print("SNP number:", len(bed))
print()


# ============================================================
# 3. GFF attribute parser
# ============================================================

def parse_attributes(attr_string):

    attrs = {}

    for item in attr_string.strip().split(";"):

        if "=" not in item:
            continue

        key, value = item.split("=", 1)

        attrs[key] = value

    return attrs


# ============================================================
# 4. Parse Gap
#
# 例如：
#
# Gap=M126 I2 M143
#
# 转换成：
#
# [
#     ("M", 126),
#     ("I", 2),
#     ("M", 143)
# ]
# ============================================================

def parse_gap(gap_string):

    if gap_string is None:
        return []

    gap_string = gap_string.strip()

    if gap_string == "":
        return []

    tokens = gap_string.split()

    result = []

    for token in tokens:

        m = re.fullmatch(r"([MID])(\d+)", token)

        if m is None:
            raise ValueError(
                f"无法解析 Gap token: {token}"
            )

        operation = m.group(1)
        length = int(m.group(2))

        result.append(
            (operation, length)
        )

    return result


# ============================================================
# 5. 读取 GFF
#
# 只保存：
#
# feature = match
#
# chromosome_to_chromosome = 1
#
# 并且 source chromosome 出现在 BED 中
# ============================================================

source_chromosomes = set(
    bed["source_chr"].unique()
)

matches_by_chr = defaultdict(list)

total_match = 0
chromosome_match = 0

with open(gff_file, "r") as f:

    for line in f:

        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        source_chr = fields[0]
        feature = fields[2]

        if feature != "match":
            continue

        total_match += 1

        # 只读取 BED 中涉及的 chromosome
        if source_chr not in source_chromosomes:
            continue

        try:

            source_start = int(fields[3])
            source_end = int(fields[4])

        except ValueError:

            continue

        source_strand = fields[6]

        attrs = parse_attributes(fields[8])

        # ----------------------------------------------------
        # 必须是 chromosome-to-chromosome
        # ----------------------------------------------------

        if attrs.get("chromosome_to_chromosome", "0") != "1":
            continue

        # ----------------------------------------------------
        # Target
        #
        # Target=NC_019468.2 56646186 56646456 +
        # ----------------------------------------------------

        if "Target" not in attrs:
            continue

        target_parts = attrs["Target"].split()

        if len(target_parts) < 3:
            continue

        target_chr = target_parts[0]

        try:

            target_start = int(target_parts[1])
            target_end = int(target_parts[2])

        except ValueError:

            continue

        if len(target_parts) >= 4:
            target_strand = target_parts[3]
        else:
            target_strand = "+"

        # ----------------------------------------------------
        # Gap
        # ----------------------------------------------------

        gap_string = attrs.get("Gap", "")

        try:

            gap = parse_gap(gap_string)

        except ValueError:

            continue

        # ----------------------------------------------------
        # reciprocal
        # ----------------------------------------------------

        reciprocity = attrs.get("reciprocity", "")

        same_unit_reciprocity = attrs.get(
            "same_unit_reciprocity",
            ""
        )

        best_on_query = attrs.get(
            "best_on_query",
            ""
        )

        best_on_subject = attrs.get(
            "best_on_subject",
            ""
        )

        # ----------------------------------------------------
        # 保存
        # ----------------------------------------------------

        match = {

            "source_chr": source_chr,

            "source_start": source_start,
            "source_end": source_end,

            "source_strand": source_strand,

            "target_chr": target_chr,

            "target_start": target_start,
            "target_end": target_end,

            "target_strand": target_strand,

            "reciprocity": reciprocity,

            "same_unit_reciprocity":
                same_unit_reciprocity,

            "best_on_query":
                best_on_query,

            "best_on_subject":
                best_on_subject,

            "gap": gap,

            "gap_string": gap_string,

            "gap_count":
                int(attrs.get("gap_count", 0)),

            "alignment_id":
                attrs.get("ID", ""),

            "num_ident":
                attrs.get("num_ident", ""),

            "num_mismatch":
                attrs.get("num_mismatch", ""),

            "pct_identity_ungap":
                attrs.get("pct_identity_ungap", "")
        }

        matches_by_chr[source_chr].append(match)

        chromosome_match += 1


print("========================================")
print("GFF")
print("========================================")
print("Total match:", total_match)
print(
    "Chromosome-to-chromosome matches:",
    chromosome_match
)
print()


# ============================================================
# 6. 检查 source / target 长度
#
# 对每一个 alignment：
#
# M -> source + length, target + length
# D -> source + length, target + 0
# I -> source + 0,      target + length
#
# 检查 Gap 与 GFF 坐标是否一致
# ============================================================

def gap_consumption(gap):

    source_consumed = 0
    target_consumed = 0

    for op, length in gap:

        if op == "M":

            source_consumed += length
            target_consumed += length

        elif op == "D":

            source_consumed += length

        elif op == "I":

            target_consumed += length

    return source_consumed, target_consumed


invalid_alignment_count = 0

for chrom in matches_by_chr:

    for match in matches_by_chr[chrom]:

        gap = match["gap"]

        if len(gap) == 0:
            continue

        source_consumed, target_consumed = \
            gap_consumption(gap)

        source_length = (
            match["source_end"]
            - match["source_start"]
            + 1
        )

        target_length = (
            match["target_end"]
            - match["target_start"]
            + 1
        )

        if source_consumed != source_length:

            invalid_alignment_count += 1

        if target_consumed != target_length:

            invalid_alignment_count += 1


print(
    "Alignment length inconsistencies:",
    invalid_alignment_count
)

print()


# ============================================================
# 7. 根据 Gap 找 SNP 对应位置
#
# 核心函数
# ============================================================

def map_snp_through_alignment(
    source_pos,
    source_start,
    source_end,
    source_strand,
    target_start,
    target_end,
    target_strand,
    gap
):

    # --------------------------------------------------------
    # 如果没有 Gap
    # --------------------------------------------------------

    if len(gap) == 0:

        if source_strand == "+":

            offset = source_pos - source_start

        else:

            offset = source_end - source_pos

        if target_strand == "+":

            target_pos = target_start + offset

        else:

            target_pos = target_end - offset

        return {
            "status": "mapped",
            "target_pos": target_pos,
            "alignment_segment": "M",
            "segment_length": 1
        }


    # --------------------------------------------------------
    # alignment 当前 source position
    #
    # 根据 source strand 决定方向
    # --------------------------------------------------------

    if source_strand == "+":

        current_source = source_start
        source_direction = 1

    else:

        current_source = source_end
        source_direction = -1


    # --------------------------------------------------------
    # alignment 当前 target position
    # --------------------------------------------------------

    if target_strand == "+":

        current_target = target_start
        target_direction = 1

    else:

        current_target = target_end
        target_direction = -1


    # --------------------------------------------------------
    # 逐段解析 Gap
    # --------------------------------------------------------

    for op, length in gap:

        # ====================================================
        # M
        #
        # Source 和 Target 一一对应
        # ====================================================

        if op == "M":

            # 当前 M 区段覆盖的 source 范围

            if source_direction == 1:

                segment_source_start = current_source
                segment_source_end = (
                    current_source + length - 1
                )

            else:

                segment_source_start = (
                    current_source - length + 1
                )

                segment_source_end = current_source


            # -----------------------------------------------
            # SNP 是否在 M 区段
            # -----------------------------------------------

            if (
                segment_source_start
                <= source_pos
                <= segment_source_end
            ):

                if source_direction == 1:

                    offset = (
                        source_pos
                        - current_source
                    )

                else:

                    offset = (
                        current_source
                        - source_pos
                    )

                target_pos = (
                    current_target
                    + target_direction * offset
                )

                return {
                    "status": "mapped",
                    "target_pos": target_pos,
                    "alignment_segment": "M",
                    "segment_length": length
                }


            # -----------------------------------------------
            # SNP 不在该 M 区段
            # -----------------------------------------------

            current_source += (
                source_direction * length
            )

            current_target += (
                target_direction * length
            )


        # ====================================================
        # D
        #
        # Source 有碱基
        # Target 没有
        #
        # 如果 SNP 落在这里：
        #
        # unmapped_gap
        # ====================================================

        elif op == "D":

            if source_direction == 1:

                segment_source_start = current_source
                segment_source_end = (
                    current_source + length - 1
                )

            else:

                segment_source_start = (
                    current_source - length + 1
                )

                segment_source_end = current_source


            if (
                segment_source_start
                <= source_pos
                <= segment_source_end
            ):

                return {
                    "status": "gap",
                    "target_pos": None,
                    "alignment_segment": "D",
                    "segment_length": length
                }


            current_source += (
                source_direction * length
            )


        # ====================================================
        # I
        #
        # Target 有额外碱基
        # Source 没有
        #
        # 所以 SNP 不可能落在 I
        #
        # 但是 target pointer 必须移动
        # ====================================================

        elif op == "I":

            current_target += (
                target_direction * length
            )


    # 如果执行到这里","说明 SNP 没有进入任何 M
    return {
        "status": "not_in_alignment",
        "target_pos": None,
        "alignment_segment": None,
        "segment_length": None
    }


# ============================================================
# 8. 给 alignment 打分
#
# 优先级：
#
# chromosome-to-chromosome
# reciprocity=3
# same_unit_reciprocity=3
# best_on_query=1
# best_on_subject=1
# gap 少
# identity 高
# ============================================================

def alignment_score(match):

    score = 0

    # chromosome-to-chromosome 已经筛选
    score += 100

    if str(match["reciprocity"]) == "3":
        score += 30

    if str(
        match["same_unit_reciprocity"]
    ) == "3":
        score += 20

    if str(
        match["best_on_query"]
    ) == "1":
        score += 10

    if str(
        match["best_on_subject"]
    ) == "1":
        score += 10

    # gap 越少越好
    score -= match["gap_count"]

    # identity
    try:

        score += float(
            match["pct_identity_ungap"]
        )

    except:

        pass

    return score


# ============================================================
# 9. Mapping 所有 SNP
# ============================================================

mapped_records = []
unmapped_records = []

for index, snp in bed.iterrows():

    source_chr = snp["source_chr"]
    source_pos = int(snp["source_pos"])

    candidates = []

    # --------------------------------------------------------
    # 找覆盖 SNP 的 alignment
    # --------------------------------------------------------

    for match in matches_by_chr.get(
        source_chr,
        []
    ):

        if (
            match["source_start"]
            <= source_pos
            <= match["source_end"]
        ):

            candidates.append(match)


    # --------------------------------------------------------
    # 没有 alignment
    # --------------------------------------------------------

    if len(candidates) == 0:

        unmapped_records.append({

            "source_chr": source_chr,
            "source_pos": source_pos,

            "status":
                "no_alignment",

            "reason":
                "no_chromosome_to_chromosome_alignment"

        })

        continue


    # --------------------------------------------------------
    # 给 candidate 排序
    # --------------------------------------------------------

    candidates = sorted(
        candidates,
        key=alignment_score,
        reverse=True
    )


    # --------------------------------------------------------
    # 尝试所有 candidates
    #
    # 如果第一条落在 D gap","
    # 仍然尝试其他 alignment
    # --------------------------------------------------------

    successful = []

    gap_hits = []

    for match in candidates:

        result = map_snp_through_alignment(

            source_pos=source_pos,

            source_start=match["source_start"],
            source_end=match["source_end"],

            source_strand=match["source_strand"],

            target_start=match["target_start"],
            target_end=match["target_end"],

            target_strand=match["target_strand"],

            gap=match["gap"]
        )

        if result["status"] == "mapped":

            successful.append(
                (match, result)
            )

        elif result["status"] == "gap":

            gap_hits.append(
                (match, result)
            )


    # ========================================================
    # 10. 有多个成功 mapping
    # ========================================================

    if len(successful) > 0:

        # 按 alignment score 排序

        successful = sorted(
            successful,
            key=lambda x:
                alignment_score(x[0]),
            reverse=True
        )

        best_match, best_result = \
            successful[0]

        best_score = alignment_score(
            best_match
        )

        # 检查是否存在相同 score 的多个 mapping

        tied = [

            x for x in successful

            if alignment_score(x[0])
            == best_score
        ]

        if len(tied) > 1:

            status = "ambiguous"

        else:

            status = "mapped"


        mapped_records.append({

            "source_chr":
                source_chr,

            "source_pos":
                source_pos,

            "target_chr":
                best_match["target_chr"],

            "target_pos":
                best_result["target_pos"],

            "target_bed_start":
                best_result["target_pos"] - 1,

            "target_bed_end":
                best_result["target_pos"],

            "source_strand":
                best_match["source_strand"],

            "target_strand":
                best_match["target_strand"],

            "mapping_status":
                status,

            "alignment_segment":
                best_result["alignment_segment"],

            "gap_count":
                best_match["gap_count"],

            "gap":
                best_match["gap_string"],

            "reciprocity":
                best_match["reciprocity"],

            "same_unit_reciprocity":
                best_match[
                    "same_unit_reciprocity"
                ],

            "best_on_query":
                best_match["best_on_query"],

            "best_on_subject":
                best_match["best_on_subject"],

            "num_ident":
                best_match["num_ident"],

            "num_mismatch":
                best_match["num_mismatch"],

            "pct_identity_ungap":
                best_match[
                    "pct_identity_ungap"
                ],

            "alignment_ID":
                best_match["alignment_id"],

            "candidate_count":
                len(candidates),

            "successful_mapping_count":
                len(successful)

        })

        continue


    # ========================================================
    # 11. 所有 alignment 都把 SNP 放进 Gap
    # ========================================================

    if len(gap_hits) > 0:

        best_match, best_result = sorted(
            gap_hits,
            key=lambda x:
                alignment_score(x[0]),
            reverse=True
        )[0]

        unmapped_records.append({

            "source_chr":
                source_chr,

            "source_pos":
                source_pos,

            "status":
                "gap",

            "reason":
                "SNP_lies_in_source_only_gap",

            "target_chr":
                best_match["target_chr"],

            "gap":
                best_match["gap_string"],

            "alignment_ID":
                best_match["alignment_id"]

        })

        continue


    # ========================================================
    # 12. 其他情况
    # ========================================================

    unmapped_records.append({

        "source_chr":
            source_chr,

        "source_pos":
            source_pos,

        "status":
            "unmapped",

        "reason":
            "alignment_found_but_coordinate_not_mapped"

    })


# ============================================================
# 13. DataFrame
# ============================================================

mapped_df = pd.DataFrame(
    mapped_records
)

unmapped_df = pd.DataFrame(
    unmapped_records
)


# ============================================================
# 14. 输出 BED
#
# 这里只输出真正 mapped 的 SNP
#
# ambiguous 暂时也输出","但 detail 中会标记
# ============================================================

if len(mapped_df) > 0:

    mapped_df[
        ["target_chr",
         "target_bed_start",
         "target_bed_end"]
    ].to_csv(

        out_bed,

        sep="\t",

        header=False,

        index=False
    )


# ============================================================
# 15. 输出 detail
# ============================================================

if len(mapped_df) > 0:

    mapped_df.to_csv(
        out_detail,
        sep="\t",
        index=False
    )


# ============================================================
# 16. 输出 unmapped
# ============================================================

if len(unmapped_df) > 0:

    unmapped_df.to_csv(
        out_unmapped,
        sep="\t",
        index=False
    )


# ============================================================
# 17. 统计
# ============================================================

print()
print("========================================")
print("Mapping result")
print("========================================")

print(
    "Total SNP:",
    len(bed)
)

print(
    "Mapped:",
    len(mapped_df)
)

print(
    "Unmapped:",
    len(unmapped_df)
)

if len(mapped_df) > 0:

    print(
        "Mapping rate:",
        f"{len(mapped_df) / len(bed) * 100:.2f}%"
    )

    print()
    print("Mapping status:")

    print(
        mapped_df[
            "mapping_status"
        ].value_counts()
    )


if len(unmapped_df) > 0:

    print()
    print("Unmapped status:")

    print(
        unmapped_df[
            "status"
        ].value_counts()
    )


print()
print("Output BED:")
print(out_bed)

print()
print("Mapping detail:")
print(out_detail)

print()
print("Unmapped:")
print(out_unmapped)

Input BED
SNP number: 2

GFF
Total match: 44958
Chromosome-to-chromosome matches: 2004

Alignment length inconsistencies: 0


Mapping result
Total SNP: 2
Mapped: 1
Unmapped: 1
Mapping rate: 50.00%

Mapping status:
mapping_status
mapped    1
Name: count, dtype: int64

Unmapped status:
status
no_alignment    1
Name: count, dtype: int64

Output BED:
/home/hnadmin/data/yangshuju/result/ovary/only_snp_v4.bed

Mapping detail:
/home/hnadmin/data/yangshuju/result/ovary/only_snp_v4_mapping_detail.tsv

Unmapped:
/home/hnadmin/data/yangshuju/result/ovary/only_snp_v4.unmapped.tsv


GCF_000298735.2 → GCF_016772045.1

In [1]:
import pandas as pd
import re
from collections import defaultdict

# ============================================================
# 1. 文件
# ============================================================

# 输入：
# GCF_000298735.2 / Oar_v2
# SNP / LD block BED
bed_file = "/home/hnadmin/data/yangshuju/result/ovary//home/hnadmin/data/yangshuju/result/ovary/gwas_v2.bed"

# 原来的 alignment GFF
gff_file = "/home/hnadmin/data/yangshuju/sample/GCF_000298735.2-GCF_016772045.1.gff"

# 输出：
# GCF_016772045.1 / Oar_v4
out_bed = "/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_v2.bed"

out_detail = "/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_v2_mapping_detail.tsv"

out_unmapped = "/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_v2.unmapped.tsv"


# ============================================================
# 2. 读取 BED
#
# 输入 BED:
#
# chromosome    start    end    GWAS_snp    R2
#
# 例如：
#
# NC_019459.2    117616308    117616309    SNP1    0.85
#
# ============================================================

bed = pd.read_csv(
    bed_file,
    sep="\t",
    header=None,
    names=[
        "source_chr",
        "bed_start",
        "bed_end",
        "GWAS_snp",
        "R2"
    ],
    dtype={
        "source_chr": str,
        "bed_start": int,
        "bed_end": int,
        "GWAS_snp": str,
        "R2": float
    }
)

# SNP 的 1-based position
bed["source_pos"] = bed["bed_start"] + 1


print("========================================")
print("Input BED")
print("========================================")
print("SNP number:", len(bed))
print()

print("Input chromosomes:")
print(
    bed["source_chr"]
    .value_counts()
    .sort_index()
)
print()


# ============================================================
# 3. GFF attribute parser
# ============================================================

def parse_attributes(attr_string):

    attrs = {}

    for item in attr_string.strip().split(";"):

        if "=" not in item:
            continue

        key, value = item.split("=", 1)

        attrs[key] = value

    return attrs


# ============================================================
# 4. Parse Gap
# ============================================================

def parse_gap(gap_string):

    if gap_string is None:
        return []

    gap_string = gap_string.strip()

    if gap_string == "":
        return []

    tokens = gap_string.split()

    result = []

    for token in tokens:

        m = re.fullmatch(
            r"([MID])(\d+)",
            token
        )

        if m is None:

            raise ValueError(
                f"无法解析 Gap token: {token}"
            )

        operation = m.group(1)
        length = int(m.group(2))

        result.append(
            (operation, length)
        )

    return result


# ============================================================
# 5. 反转 Gap
#
# 原方向：
#
# M = source 和 target 都消耗
# D = source 有","target 没有
# I = target 有","source 没有
#
# 反向以后：
#
# M -> M
# D -> I
# I -> D
# ============================================================

def reverse_gap(gap):

    reversed_gap = []

    for op, length in gap:

        if op == "M":

            new_op = "M"

        elif op == "D":

            new_op = "I"

        elif op == "I":

            new_op = "D"

        else:

            raise ValueError(
                f"未知 Gap operation: {op}"
            )

        reversed_gap.append(
            (new_op, length)
        )

    return reversed_gap


# ============================================================
# 6. 读取 GFF
#
# 注意：
#
# 原 GFF：
#
# source = GCF_016772045.1
# target = GCF_000298735.2
#
# 这里我们反过来：
#
# source = GCF_000298735.2
# target = GCF_016772045.1
# ============================================================

source_chromosomes = set(
    bed["source_chr"].unique()
)

matches_by_chr = defaultdict(list)

total_match = 0
chromosome_match = 0


with open(gff_file, "r") as f:

    for line in f:

        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        # ----------------------------------------------------
        # 原始 GFF source
        # ----------------------------------------------------

        gff_source_chr = fields[0]

        feature = fields[2]

        if feature != "match":
            continue

        total_match += 1

        source_start_original = int(fields[3])
        source_end_original = int(fields[4])

        source_strand_original = fields[6]

        attrs = parse_attributes(fields[8])

        # ----------------------------------------------------
        # chromosome-to-chromosome
        # ----------------------------------------------------

        if attrs.get(
            "chromosome_to_chromosome",
            "0"
        ) != "1":

            continue

        # ----------------------------------------------------
        # Target
        #
        # Target=NC_xxxxx 12345 67890 +
        # ----------------------------------------------------

        if "Target" not in attrs:
            continue

        target_parts = attrs["Target"].split()

        if len(target_parts) < 3:
            continue

        gff_target_chr = target_parts[0]

        target_start_original = int(
            target_parts[1]
        )

        target_end_original = int(
            target_parts[2]
        )

        if len(target_parts) >= 4:

            target_strand_original = (
                target_parts[3]
            )

        else:

            target_strand_original = "+"


        # ----------------------------------------------------
        # 关键：
        #
        # 现在 BED 是 GCF_000298735.2
        #
        # 因此原始 GFF target 必须出现在 BED chromosome 中
        # ----------------------------------------------------

        if (
            gff_target_chr
            not in source_chromosomes
        ):

            continue


        # ----------------------------------------------------
        # 原始 Gap
        # ----------------------------------------------------

        gap_string = attrs.get(
            "Gap",
            ""
        )

        try:

            gap_original = parse_gap(
                gap_string
            )

        except ValueError:

            continue


        # ----------------------------------------------------
        # 反向 Gap
        # ----------------------------------------------------

        gap_reverse = reverse_gap(
            gap_original
        )


        # ----------------------------------------------------
        # 原始 alignment：
        #
        # GCF_016772045
        #       ↓
        # GCF_000298735
        #
        # 现在构造成：
        #
        # GCF_000298735
        #       ↓
        # GCF_016772045
        # ----------------------------------------------------

        match = {

            # ==============================
            # 反向 source
            # ==============================

            "source_chr":
                gff_target_chr,

            "source_start":
                target_start_original,

            "source_end":
                target_end_original,

            "source_strand":
                target_strand_original,


            # ==============================
            # 反向 target
            # ==============================

            "target_chr":
                gff_source_chr,

            "target_start":
                source_start_original,

            "target_end":
                source_end_original,

            "target_strand":
                source_strand_original,


            # ==============================
            # metadata
            # ==============================

            "reciprocity":
                attrs.get(
                    "reciprocity",
                    ""
                ),

            "same_unit_reciprocity":
                attrs.get(
                    "same_unit_reciprocity",
                    ""
                ),

            "best_on_query":
                attrs.get(
                    "best_on_query",
                    ""
                ),

            "best_on_subject":
                attrs.get(
                    "best_on_subject",
                    ""
                ),

            # ==============================
            # reverse Gap
            # ==============================

            "gap":
                gap_reverse,

            "gap_string":
                " ".join(
                    [
                        f"{op}{length}"
                        for op, length
                        in gap_reverse
                    ]
                ),

            "gap_count":
                int(
                    attrs.get(
                        "gap_count",
                        0
                    )
                ),

            "alignment_id":
                attrs.get(
                    "ID",
                    ""
                ),

            "num_ident":
                attrs.get(
                    "num_ident",
                    ""
                ),

            "num_mismatch":
                attrs.get(
                    "num_mismatch",
                    ""
                ),

            "pct_identity_ungap":
                attrs.get(
                    "pct_identity_ungap",
                    ""
                )
        }


        matches_by_chr[
            gff_target_chr
        ].append(match)

        chromosome_match += 1


print("========================================")
print("GFF")
print("========================================")

print(
    "Total match:",
    total_match
)

print(
    "Reverse chromosome-to-chromosome matches:",
    chromosome_match
)

print()


# ============================================================
# 7. 检查 reverse Gap 长度
# ============================================================

def gap_consumption(gap):

    source_consumed = 0
    target_consumed = 0

    for op, length in gap:

        if op == "M":

            source_consumed += length
            target_consumed += length

        elif op == "D":

            source_consumed += length

        elif op == "I":

            target_consumed += length

    return (
        source_consumed,
        target_consumed
    )


invalid_alignment_count = 0


for chrom in matches_by_chr:

    for match in matches_by_chr[chrom]:

        gap = match["gap"]

        if len(gap) == 0:
            continue

        source_consumed, target_consumed = \
            gap_consumption(gap)

        source_length = (
            match["source_end"]
            - match["source_start"]
            + 1
        )

        target_length = (
            match["target_end"]
            - match["target_start"]
            + 1
        )

        if source_consumed != source_length:

            invalid_alignment_count += 1

        if target_consumed != target_length:

            invalid_alignment_count += 1


print(
    "Reverse alignment length inconsistencies:",
    invalid_alignment_count
)

print()


# ============================================================
# 8. SNP mapping function
#
# 这里直接沿用你原来的核心算法
# ============================================================

def map_snp_through_alignment(
    source_pos,
    source_start,
    source_end,
    source_strand,
    target_start,
    target_end,
    target_strand,
    gap
):

    # --------------------------------------------------------
    # 没有 Gap
    # --------------------------------------------------------

    if len(gap) == 0:

        if source_strand == "+":

            offset = (
                source_pos
                - source_start
            )

        else:

            offset = (
                source_end
                - source_pos
            )

        if target_strand == "+":

            target_pos = (
                target_start
                + offset
            )

        else:

            target_pos = (
                target_end
                - offset
            )

        return {

            "status":
                "mapped",

            "target_pos":
                target_pos,

            "alignment_segment":
                "M",

            "segment_length":
                1
        }


    # --------------------------------------------------------
    # source direction
    # --------------------------------------------------------

    if source_strand == "+":

        current_source = source_start
        source_direction = 1

    else:

        current_source = source_end
        source_direction = -1


    # --------------------------------------------------------
    # target direction
    # --------------------------------------------------------

    if target_strand == "+":

        current_target = target_start
        target_direction = 1

    else:

        current_target = target_end
        target_direction = -1


    # --------------------------------------------------------
    # 遍历 Gap
    # --------------------------------------------------------

    for op, length in gap:

        # ====================================================
        # M
        # ====================================================

        if op == "M":

            if source_direction == 1:

                segment_source_start = \
                    current_source

                segment_source_end = (
                    current_source
                    + length
                    - 1
                )

            else:

                segment_source_start = (
                    current_source
                    - length
                    + 1
                )

                segment_source_end = \
                    current_source


            if (
                segment_source_start
                <= source_pos
                <= segment_source_end
            ):

                if source_direction == 1:

                    offset = (
                        source_pos
                        - current_source
                    )

                else:

                    offset = (
                        current_source
                        - source_pos
                    )

                target_pos = (
                    current_target
                    + target_direction
                    * offset
                )

                return {

                    "status":
                        "mapped",

                    "target_pos":
                        target_pos,

                    "alignment_segment":
                        "M",

                    "segment_length":
                        length
                }


            current_source += (
                source_direction
                * length
            )

            current_target += (
                target_direction
                * length
            )


        # ====================================================
        # D
        # ====================================================

        elif op == "D":

            if source_direction == 1:

                segment_source_start = \
                    current_source

                segment_source_end = (
                    current_source
                    + length
                    - 1
                )

            else:

                segment_source_start = (
                    current_source
                    - length
                    + 1
                )

                segment_source_end = \
                    current_source


            if (
                segment_source_start
                <= source_pos
                <= segment_source_end
            ):

                return {

                    "status":
                        "gap",

                    "target_pos":
                        None,

                    "alignment_segment":
                        "D",

                    "segment_length":
                        length
                }


            current_source += (
                source_direction
                * length
            )


        # ====================================================
        # I
        # ====================================================

        elif op == "I":

            current_target += (
                target_direction
                * length
            )


    return {

        "status":
            "not_in_alignment",

        "target_pos":
            None,

        "alignment_segment":
            None,

        "segment_length":
            None
    }


# ============================================================
# 9. Alignment scoring
# ============================================================

def alignment_score(match):

    score = 0

    score += 100

    if str(
        match["reciprocity"]
    ) == "3":

        score += 30


    if str(
        match["same_unit_reciprocity"]
    ) == "3":

        score += 20


    if str(
        match["best_on_query"]
    ) == "1":

        score += 10


    if str(
        match["best_on_subject"]
    ) == "1":

        score += 10


    score -= match["gap_count"]


    try:

        score += float(
            match[
                "pct_identity_ungap"
            ]
        )

    except:

        pass


    return score


# ============================================================
# 10. Mapping SNP
# ============================================================

mapped_records = []
unmapped_records = []


for index, snp in bed.iterrows():

    source_chr = snp[
        "source_chr"
    ]

    source_pos = int(
        snp[
            "source_pos"
        ]
    )


    # --------------------------------------------------------
    # 找所有覆盖 SNP 的 alignment
    # --------------------------------------------------------

    candidates = []


    for match in matches_by_chr.get(
        source_chr,
        []
    ):

        if (
            match["source_start"]
            <= source_pos
            <= match["source_end"]
        ):

            candidates.append(
                match
            )


    # --------------------------------------------------------
    # 没有 alignment
    # --------------------------------------------------------

    if len(candidates) == 0:

        unmapped_records.append({

            "source_chr":
                source_chr,

            "source_pos":
                source_pos,

            "GWAS_snp":
                snp["GWAS_snp"],

            "R2":
                snp["R2"],

            "status":
                "no_alignment",

            "reason":
                "no_reverse_alignment"

        })

        continue


    # --------------------------------------------------------
    # 按 alignment score 排序
    # --------------------------------------------------------

    candidates = sorted(
        candidates,
        key=alignment_score,
        reverse=True
    )


    successful = []
    gap_hits = []


    # --------------------------------------------------------
    # 尝试所有 alignment
    # --------------------------------------------------------

    for match in candidates:

        result = map_snp_through_alignment(

            source_pos=source_pos,

            source_start=
                match[
                    "source_start"
                ],

            source_end=
                match[
                    "source_end"
                ],

            source_strand=
                match[
                    "source_strand"
                ],

            target_start=
                match[
                    "target_start"
                ],

            target_end=
                match[
                    "target_end"
                ],

            target_strand=
                match[
                    "target_strand"
                ],

            gap=
                match[
                    "gap"
                ]
        )


        if result[
            "status"
        ] == "mapped":

            successful.append(
                (
                    match,
                    result
                )
            )


        elif result[
            "status"
        ] == "gap":

            gap_hits.append(
                (
                    match,
                    result
                )
            )


    # ========================================================
    # 11. 成功 mapping
    # ========================================================

    if len(successful) > 0:

        successful = sorted(
            successful,
            key=lambda x:
                alignment_score(
                    x[0]
                ),
            reverse=True
        )


        best_match, best_result = \
            successful[0]


        best_score = alignment_score(
            best_match
        )


        tied = [

            x for x in successful

            if alignment_score(
                x[0]
            ) == best_score

        ]


        if len(tied) > 1:

            status = "ambiguous"

        else:

            status = "mapped"


        mapped_records.append({

            "source_chr":
                source_chr,

            "source_pos":
                source_pos,

            "GWAS_snp":
                snp["GWAS_snp"],

            "R2":
                snp["R2"],

            "target_chr":
                best_match[
                    "target_chr"
                ],

            "target_pos":
                best_result[
                    "target_pos"
                ],

            "target_bed_start":
                best_result[
                    "target_pos"
                ] - 1,

            "target_bed_end":
                best_result[
                    "target_pos"
                ],

            "source_strand":
                best_match[
                    "source_strand"
                ],

            "target_strand":
                best_match[
                    "target_strand"
                ],

            "mapping_status":
                status,

            "alignment_segment":
                best_result[
                    "alignment_segment"
                ],

            "gap_count":
                best_match[
                    "gap_count"
                ],

            "gap":
                best_match[
                    "gap_string"
                ],

            "reciprocity":
                best_match[
                    "reciprocity"
                ],

            "same_unit_reciprocity":
                best_match[
                    "same_unit_reciprocity"
                ],

            "best_on_query":
                best_match[
                    "best_on_query"
                ],

            "best_on_subject":
                best_match[
                    "best_on_subject"
                ],

            "num_ident":
                best_match[
                    "num_ident"
                ],

            "num_mismatch":
                best_match[
                    "num_mismatch"
                ],

            "pct_identity_ungap":
                best_match[
                    "pct_identity_ungap"
                ],

            "alignment_ID":
                best_match[
                    "alignment_id"
                ],

            "candidate_count":
                len(candidates),

            "successful_mapping_count":
                len(successful)

        })

        continue


    # ========================================================
    # 12. SNP 落在 Gap
    # ========================================================

    if len(gap_hits) > 0:

        best_match, best_result = sorted(
            gap_hits,
            key=lambda x:
                alignment_score(
                    x[0]
                ),
            reverse=True
        )[0]


        unmapped_records.append({

            "source_chr":
                source_chr,

            "source_pos":
                source_pos,

            "GWAS_snp":
                snp["GWAS_snp"],

            "R2":
                snp["R2"],

            "status":
                "gap",

            "reason":
                "SNP_lies_in_source_only_gap",

            "target_chr":
                best_match[
                    "target_chr"
                ],

            "gap":
                best_match[
                    "gap_string"
                ],

            "alignment_ID":
                best_match[
                    "alignment_id"
                ]

        })

        continue


    # ========================================================
    # 13. 其他 unmapped
    # ========================================================

    unmapped_records.append({

        "source_chr":
            source_chr,

        "source_pos":
            source_pos,

        "GWAS_snp":
            snp["GWAS_snp"],

        "R2":
            snp["R2"],

        "status":
            "unmapped",

        "reason":
            "alignment_found_but_coordinate_not_mapped"

    })


# ============================================================
# 14. DataFrame
# ============================================================

mapped_df = pd.DataFrame(
    mapped_records
)

unmapped_df = pd.DataFrame(
    unmapped_records
)


# ============================================================
# 15. 输出 BED
#
# GCF_016772045.1 / Oar_v4
#
# chromosome
# start
# end
#
# 仍然是 0-based half-open
# ============================================================

if len(mapped_df) > 0:

    mapped_df[
        [
            "target_chr",
            "target_bed_start",
            "target_bed_end"
        ]
    ].to_csv(

        out_bed,

        sep="\t",

        header=False,

        index=False
    )


# ============================================================
# 16. 输出 mapping detail
# ============================================================

if len(mapped_df) > 0:

    mapped_df.to_csv(
        out_detail,
        sep="\t",
        index=False
    )


# ============================================================
# 17. 输出 unmapped
# ============================================================

if len(unmapped_df) > 0:

    unmapped_df.to_csv(
        out_unmapped,
        sep="\t",
        index=False
    )


# ============================================================
# 18. 统计
# ============================================================

print()
print("========================================")
print("Reverse Mapping Result")
print("========================================")

print(
    "Input SNP:",
    len(bed)
)

print(
    "Mapped:",
    len(mapped_df)
)

print(
    "Unmapped:",
    len(unmapped_df)
)


if len(mapped_df) > 0:

    print(
        "Mapping rate:",
        f"{len(mapped_df) / len(bed) * 100:.2f}%"
    )

    print()

    print(
        "Mapping status:"
    )

    print(
        mapped_df[
            "mapping_status"
        ].value_counts()
    )


if len(unmapped_df) > 0:

    print()

    print(
        "Unmapped status:"
    )

    print(
        unmapped_df[
            "status"
        ].value_counts()
    )


print()
print("Output BED:")
print(out_bed)

print()
print("Mapping detail:")
print(out_detail)

print()
print("Unmapped:")
print(out_unmapped)

Input BED
SNP number: 23

Input chromosomes:
source_chr
NC_019459.2     3
NC_019469.2     4
NC_019470.2    10
NC_019475.2     6
Name: count, dtype: int64

GFF
Total match: 44958
Reverse chromosome-to-chromosome matches: 3241

Reverse alignment length inconsistencies: 0


Reverse Mapping Result
Input SNP: 23
Mapped: 16
Unmapped: 7
Mapping rate: 69.57%

Mapping status:
mapping_status
mapped    16
Name: count, dtype: int64

Unmapped status:
status
no_alignment    6
gap             1
Name: count, dtype: int64

Output BED:
/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_v2.bed

Mapping detail:
/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_v2_mapping_detail.tsv

Unmapped:
/home/hnadmin/data/yangshuju/result/ovary/litter_size_GWAS_v2.unmapped.tsv


In [ ]:
bedtools intersect -a HY-ovary-H3K27ac.specific.bed -b /home/hnadmin/data/yangshuju/result/ovary/gwas_LD_Oar_v4.bed -wa -wb > HY-ovary-H3K27ac.specific.gwas.bed
bedtools intersect -a HTY-ovary-H3K27ac.specific.bed -b /home/hnadmin/data/yangshuju/result/ovary/gwas_LD_Oar_v4.bed -wa -wb > HTY-ovary-H3K27ac.specific.gwas.bed
bedtools intersect -a HY-ovary-H3K4me3.specific.bed -b /home/hnadmin/data/yangshuju/result/ovary/gwas_LD_Oar_v4.bed -wa -wb > HY-ovary-H3K4me3.specific.gwas.bed
bedtools intersect -a HTY-ovary-H3K4me3.specific.bed -b /home/hnadmin/data/yangshuju/result/ovary/gwas_LD_Oar_v4.bed -wa -wb > HTY-ovary-H3K4me3.specific.gwas.bed


In [ ]:
plink2 \
  --vcf SNP300.vcf \
  --ld-snp-list litter_size_GWAS_v2.bed \
  --r2-unphased \
  --ld-window-kb 1000 \
  --ld-window 999999 \
  --ld-window-r2 0.8 \
  --out litter_size_GWAS_v2_0.8_LD \
  --threads 64

In [ ]:
plink2 \
  --vcf SNP300.vcf \
  --ld-snp-list litter_size_GWAS_v2.bed \
  --r2-unphased \
  --ld-window-kb 1000 \
  --ld-window 999999 \
  --ld-window-r2 0.6 \
  --out litter_size_GWAS_v2_0.6_LD \
  --threads 64

In [1]:
import pandas as pd
gwas = pd.read_csv('/home/hnadmin/data/yangshuju/result/ovary/gwas_LD_Oar_v4_mapping_detail.tsv',sep='\t')
ld_snp = [14363011,14363055,14363170,14363192,14363215,14363237,14361050,14361087]
gwas = gwas[gwas['target_pos'].isin(ld_snp)]
snp_list = list(gwas['source_chr']+':' + gwas['source_pos'].astype(str))
snp_ld = pd.read_csv('/home/hnadmin/data/yangshuju/result/ovary/NC_056062.1_14474944_LD.tsv',sep='\t',header=None)
snp_ld = snp_ld[snp_ld[5].isin(snp_list)].reset_index()
#gwas = gwas[['target_chr','target_pos']]
snp_ld =snp_ld[[5,6]]
snp_ld.columns = ['target_pos','r2']
gwas['target_pos'] = gwas['target_chr'].astype(str)+':'+gwas['target_pos'].astype(str)
gwas['source_pos'] = gwas['source_chr'].astype(str)+':'+gwas['source_pos'].astype(str)
# gwas =gwas[['source_pos','target_pos']]
# gwas_snp_ld = pd.merge(gwas,snp_ld,left_on='source_pos',right_on='target_pos',how='left')
# gwas_snp_ld['orignal_snp'] = 'NC_056062.1:14474944'
# gwas_snp_ld = gwas_snp_ld[['orignal_snp','source_pos','target_pos_x','r2']]
# gwas_snp_ld.columns = ['orignal_snp','LD_SNP_v2','LD_SNP_V4','r2']
# gwas_snp_ld['orignal_snp_V4'] ='NC_019466.2:14474944'
# gwas_snp_ld.to_csv('NC_056062.1_14474944_LD.tsv',sep='\t',index=False)

KeyError: 5

motifBreakR

In [4]:
library(JASPAR2024)
library(TFBSTools)

opts <- list(
    collection = "CORE",
    tax_group = "vertebrates"
)

pfm <- getMatrixSet(
    JASPAR2024,
    opts
)

length(pfm)

Loading required package: BiocFileCache

Loading required package: dbplyr



ERROR: Error in library(TFBSTools): there is no package called ‘TFBSTools’
